In [1]:
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
print("project root:", PROJECT_ROOT)

project root: C:\Users\Anshul\Agentic_RAG


In [2]:
import fitz
import re

def clean_text(text):
    text = re.sub(r"-\n", "", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"\s{2,}", " ", text)
    return text.strip()

def chunk_text(text, chunk_size=512, overlap=50):
    target_words = int(chunk_size * 0.75)
    overlap_words = int(overlap * 0.75)
    
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = start + target_words
        chunk = " ".join(words[start:end])
        if len(chunk.strip()) > 50:
            chunks.append(chunk)
        start += target_words - overlap_words
    
    return chunks

In [3]:
# doing it for the entire dataset
papers_dir = os.path.join('..', 'data', 'papers')
all_chunks = []

for filename in os.listdir(papers_dir):
    if not filename.endswith('.pdf'):
        continue

    path = os.path.join(papers_dir, filename)
    doc = fitz.open(path)
    paper_chunks = []
    
    for page_num in range(len(doc)):
        text = doc[page_num].get_text('text')
        cleaned = clean_text(text)

        if len(cleaned) > 100:
            chunks = chunk_text(cleaned)
            for chunk in chunks:
                paper_chunks.append(
                    {
                        'text': chunk,
                        'source': filename,
                        'page_num': page_num + 1
                    }
                )

    all_chunks.append({'file': filename, 'chunks': paper_chunks})
    print(f'{filename}: {len(paper_chunks)} chunks from {len(doc)} pages')

total = sum(len(p['chunks']) for p in all_chunks)
print(f'\nTotal chunks across all papers: {total}')
                

attention_is_all_you_need.pdf: 26 chunks from 15 pages
concrete_problems_ai_safety.pdf: 61 chunks from 29 pages
constitutional_ai.pdf: 70 chunks from 34 pages
reward_hacking_survey.pdf: 30 chunks from 19 pages
rlhf_christiano.pdf: 34 chunks from 17 pages
scalable_oversight_debate.pdf: 48 chunks from 24 pages

Total chunks across all papers: 269


# Chromadb initialisation

In [4]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

CHROMA_DB_PATH = os.path.join(PROJECT_ROOT, 'chroma_db')

client = chromadb.PersistentClient(path = CHROMA_DB_PATH)
embed_fn = SentenceTransformerEmbeddingFunction(model_name = 'all-MiniLM-L6-v2')

collection = client.get_or_create_collection(
    name = 'ai_safety_papers',
    embedding_function = embed_fn,
    metadata = {'hnsw:space': 'cosine'}
)

print('collection ready:', collection.name)
print('documents currently in collection:', collection.count())

D:\Anaconda\envs\langchain_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2144.90it/s]


collection ready: ai_safety_papers
documents currently in collection: 0


In [13]:
import hashlib

PAPER_METADATA = {
    "attention_is_all_you_need.pdf":    {"title": "Attention Is All You Need", "authors": "Vaswani et al.", "year": 2017},
    "concrete_problems_ai_safety.pdf":  {"title": "Concrete Problems in AI Safety", "authors": "Amodei et al.", "year": 2016},
    "constitutional_ai.pdf":            {"title": "Constitutional AI", "authors": "Bai et al.", "year": 2022},
    "reward_hacking_survey.pdf":        {"title": "Reward Hacking Survey", "authors": "Skalse et al.", "year": 2022},
    "rlhf_christiano.pdf":              {"title": "Deep RL from Human Preferences", "authors": "Christiano et al.", "year": 2017},
    "scalable_oversight_debate.pdf":    {"title": "AI Safety via Debate", "authors": "Irving et al.", "year": 2018},
}

ids, documents, metadatas = [], [], []

for paper in all_chunks:
    # print(paper)
    # break

    filename = paper['file']
    meta = PAPER_METADATA.get(filename, {})

    for chunk in paper['chunks']:
        raw_id = f"{filename}::p{chunk['page_num']}::c{chunk['text'][:30]}"
        chunk_id = hashlib.md5(raw_id.encode()).hexdigest()

        ids.append(chunk_id)
        documents.append(chunk['text'])
        metadatas.append({
            'source': filename,
            'title': meta.get('title', filename),
            'authors': meta.get('authors', ''),
            'year': meta.get('year', 0),
            'page_num': chunk['page_num']
        })
        # print(chunk, type(chunk))
        # break

batch_size = 50
for i in range(0, len(ids), batch_size):
    collection.upsert(
        ids = ids[i:i+batch_size],
        documents = documents[i:i+batch_size],
        metadatas = metadatas[i:i+batch_size]
    )

    print(f"inserted chunks {i} to {min(i+batch_size, len(ids))}")

print(f'\ntotal in DB: {collection.count()}')

inserted chunks 0 to 50
inserted chunks 50 to 100
inserted chunks 100 to 150
inserted chunks 150 to 200
inserted chunks 200 to 250
inserted chunks 250 to 269

total in DB: 269


In [14]:

results = collection.query(
    query_texts=["what is reward hacking?"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for i in range(3):
    meta = results["metadatas"][0][i]
    score = round(1 - results["distances"][0][i], 4)  # convert distance to similarity
    text = results["documents"][0][i][:200]
    
    print(f"--- Result {i+1} ---")
    print(f"source : {meta['title']} (page {meta['page_num']})")
    print(f"score  : {score}")
    print(f"text   : {text}")
    print()

--- Result 1 ---
source : Concrete Problems in AI Safety (page 10)
score  : 0.6267
text   : planning to replace the reward function. (Much like how a human would probably “enjoy” taking addictive substances once they do, but not want to be an addict.) Similar ideas are explored in [50, 71]. 

--- Result 2 ---
source : Reward Hacking Survey (page 2)
score  : 0.6018
text   : sensor reading at test time in a 10x10 navigation grid world. Leike et al. (2017) show examples of reward hacking in a 3x3 boat race and a 5x7 tomato watering grid world. Everitt et al. (2017) theoret

--- Result 3 ---
source : Reward Hacking Survey (page 1)
score  : 0.58
text   : user satisfaction (Stray, 2020), as well as to recommended extreme political content to users (Ribeiro et al., 2020). Addressing reward hacking is a ﬁrst step towards developing human-aligned RL agent

